# QC: what the hit calls rest on

Every number here is read from `results/tables/`, written by `scripts/02`-`04`.
The README quotes three of them; this notebook is where they come from and what
they are worth.

Four questions:

1. Is the infected / bystander split real, or a sequencing-depth artefact?
2. Did the perturbations actually knock their targets down?
3. Do two independently cloned vectors for the same gene agree?
4. Is the 10x channel a confounder?

In [1]:
import sys, pandas as pd, numpy as np
sys.path.insert(0, "../scripts")
import _common as C
pd.set_option("display.width", 200)

## 1. Infection calling

Four rules, three of them threshold-free.

In [2]:
C.read_tbl("03_classifier_agreement.tsv").round(4)

,rule,infected_frac,borderline_frac,ref_called_infected,agree_with_mixture
0,mixture (primary),0.2918,0.0967,0.1461,1.0000
1,ambient per-gene,0.3074,0.0000,0.1603,0.9926
2,fixed viral_frac > 0.009,0.3219,0.0000,0.1560,0.9994
3,"published rule, literal (mean + 2 SD of refere...",0.0624,0.0000,0.1163,0.7460


The first three agree to within 0.7%. The fourth is the paper's own rule taken
literally: it needs a guide-free uninfected reference, and the reference that can
be rebuilt from the deposited matrix is itself 11.6% infected, so the threshold
comes out far too high. That is a limitation of the deposited data, not evidence
against the published rule.

In [3]:
C.read_tbl("03_depth_sensitivity.tsv").round(4)

,downsample_frac,infected_frac,agree_with_full_depth
0,1.00,0.3219,0.9994
1,0.75,0.3225,0.9994
2,0.50,0.3231,0.9994
3,0.25,0.3218,0.9987
4,0.10,0.3224,0.9955


Downsampling the whole library tenfold changes 0.45% of calls, so the split is
not depth-driven - which matters, because infected cells are the shallow ones.

In [4]:
C.read_tbl("03_host_umi_matched_depth.tsv").round(1)

,bin,borderline,infected,uninfected
0,"(471.999, 3862.1]",3071.5,1651.0,1807.0
1,"(3862.1, 6632.2]",5358.0,2719.0,5707.0
2,"(6632.2, 9705.3]",8092.5,3326.0,8486.0
3,"(9705.3, 13696.4]",11457.0,4355.5,12005.0
4,"(13696.4, 18378.0]",15582.0,7374.0,16125.0
5,"(18378.0, 22613.6]",19830.5,15909.0,20513.0
6,"(22613.6, 26781.0]",24120.5,20902.0,24613.5
7,"(26781.0, 31702.6]",28249.0,25447.5,29030.0
8,"(31702.6, 39278.5]",33064.0,31381.0,34833.0
9,"(39278.5, 91635.0]",43752.0,41216.5,46056.0


Within bins of equal total library size, infected cells still carry far fewer
host transcripts. The shutoff is compositional, not a sequencing artefact.

## 2. On-target knockdown

In [5]:
C.read_tbl("04_knockdown_summary.tsv").round(3)

,metric,value
0,element_gene_pairs_tested,228.000
1,pairs_with_20plus_bystander_cells,197.000
2,median_knockdown_pseudobulk,0.884
3,median_knockdown_mean_of_cpm,0.910
4,frac_above_50pct_kd,0.868
5,frac_above_75pct_kd,0.756
6,frac_above_90pct_kd,0.457
7,n_negative_kd,4.000


In [6]:
kd = C.read_tbl("04_knockdown.tsv")
b = kd[(kd.population == "bystander") & (kd.n_cells >= 20)]
b.nsmallest(10, "kd_pseudobulk")[["element", "target_gene", "n_cells", "kd_pseudobulk"]].round(3)

,element,target_gene,n_cells,kd_pseudobulk
340,CRISPRi_173_ACE2_FURIN,FURIN,142.0,-0.190
157,CRISPRi_081_MEPCE,MEPCE,64.0,-0.044
145,CRISPRi_075_LARP1,LARP1,84.0,-0.007
344,CRISPRi_174_TMPRSS2_FURIN,FURIN,184.0,-0.006
111,CRISPRi_058_FURIN,FURIN,82.0,0.057
338,CRISPRi_173_ACE2_FURIN,ACE2,142.0,0.086
422,CRISPRi_217_C19orf66,C19orf66,116.0,0.179
59,CRISPRi_031_CDK6,CDK6,73.0,0.228
251,CRISPRi_129_TLE1,TLE1,48.0,0.246
0,CRISPRi_001_ACE2,ACE2,113.0,0.250


FURIN, MEPCE and LARP1 do not knock down. Any phenotype, or absence of one, for
those elements is uninterpretable, and the FURIN element is used in the paper's
protease-redundancy argument.

## 3. Construct concordance

The two categories must not be pooled.

In [7]:
C.read_tbl("04_construct_concordance_summary.tsv").round(3)

,pair_type,n_genes,pearson_r,spearman_rho,median_abs_difference
0,independent_vectors,12,0.548,0.356,0.148
1,more_than_two_elements,5,0.975,1.000,0.048
2,single_vs_combination,24,0.554,0.197,0.089


Only `independent_vectors` - 12 genes carried by two separately cloned
single-guide vectors - is a reagent-to-reagent comparison. It gives r = 0.55.
Pooling it with pairs that share guide design gives r = 0.74, which is the number
an earlier draft of this repository reported in error.

## 4. Lane structure

In [8]:
C.read_tbl("03_state_summary.tsv").round(4)

,gem_group,borderline,infected,uninfected,n_cells
0,1,0.0714,0.2405,0.6881,3725
1,2,0.0874,0.2488,0.6638,3673
2,3,0.0684,0.2442,0.6874,3346
3,4,0.0700,0.2524,0.6776,3558
4,5,0.1320,0.3447,0.5233,3264
5,6,0.1184,0.3393,0.5423,3557
6,7,0.1119,0.3333,0.5547,3243
7,8,0.1183,0.3407,0.5410,3516


Channels 5-8 carry about 1.4x the infection rate of channels 1-4, matching the
paper's pulsed and non-pulsed conditions. Lane is a stratum in every model and
every null downstream.